In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import re
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords', quiet=True) 

True

Import the dataset and rename the columns

In [2]:
df = pd.read_csv(r"C:\Rishikha\Projects\SMS_Spam_Detection\spam_sms.csv", encoding="latin-1")
df.columns = ['label', 'text']  

In [3]:
df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
print(df.shape)
print(df.columns)

(5572, 2)
Index(['label', 'text'], dtype='object')


Count the number of spam and ham messages

In [5]:
df['label'].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

In [6]:
# Convert labels to 0 (ham) and 1 (spam)
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

In [7]:
# Clean text function
def clean_text(text):
    text = text.lower()  # Coverted to lowercase
    text = re.sub(r'\W', ' ', text)  # Remove special characters
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text
df['text'] = df['text'].apply(clean_text)

In [8]:
# Split data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.2, random_state=42)
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(4457,)
(1115,)
(4457,)
(1115,)


In [9]:
max_vocab_size = 10000  # Vocabulary size
max_length = 100  # Sequence length

# Tokenize text
tokenizer = Tokenizer(num_words=max_vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(x_train)

x_train_seq = tokenizer.texts_to_sequences(x_train)
x_test_seq = tokenizer.texts_to_sequences(x_test)

# Pad sequences
x_train_pad = pad_sequences(x_train_seq, maxlen=max_length, padding='post', truncating='post')
x_test_pad = pad_sequences(x_test_seq, maxlen=max_length, padding='post', truncating='post')

In [10]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(sampling_strategy=0.5, random_state=42)  # Increase spam examples
x_train_resampled, y_train_resampled = smote.fit_resample(x_train_pad, y_train)

In [11]:
embedding_dim = 64  # Feature vector size

model = Sequential([
    Embedding(input_dim=max_vocab_size, output_dim=embedding_dim), 
    GRU(64, return_sequences=False),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

In [12]:
# Build model before calling summary
model.build(input_shape=(None, max_length))
model.compile(optimizer='adam', loss='binary_crossentropy',metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 665,025 (2.54 MB)

 Trainable params: 665,025 (2.54 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
history = model.fit(x_train_resampled, y_train_resampled, epochs=10, batch_size=32, validation_data=(x_test_pad, y_test))

Epoch 1/10
181/181 ━━━━━━━━━━━━━━━━━━━━ 8s 34ms/step - accuracy: 0.6661 - loss: 0.6403 - val_accuracy: 0.8655 - val_loss: 0.4754
Epoch 2/10
181/181 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.6667 - loss: 0.6389 - val_accuracy: 0.8655 - val_loss: 0.4981
Epoch 3/10
181/181 ━━━━━━━━━━━━━━━━━━━━ 5s 28ms/step - accuracy: 0.6667 - loss: 0.6378 - val_accuracy: 0.8655 - val_loss: 0.4721
Epoch 4/10
181/181 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.6667 - loss: 0.6379 - val_accuracy: 0.8655 - val_loss: 0.4923
Epoch 5/10
181/181 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - accuracy: 0.6667 - loss: 0.6386 - val_accuracy: 0.8655 - val_loss: 0.4900
Epoch 6/10
181/181 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.6667 - loss: 0.6374 - val_accuracy: 0.8655 - val_loss: 0.5131
Epoch 7/10
181/181 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.6667 - loss: 0.6388 - val_accuracy: 0.8655 - val_loss: 0.4737
Epoch 8/10
181/181 ━━━━━━━━━━━━━━━━━━━━ 12s 37ms/step - accuracy: 0.6667 - loss: 0.6382 - val_acc

In [14]:
loss, accuracy = model.evaluate(x_test_pad, y_test)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8655 - loss: 0.5140
Test Accuracy: 86.55%


In [15]:
from sklearn.metrics import classification_report
y_pred = (model.predict(x_test_pad) > 0.5).astype("int32")
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam'], zero_division=0))

35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step
              precision    recall  f1-score   support

         Ham       0.87      1.00      0.93       965
        Spam       0.00      0.00      0.00       150

    accuracy                           0.87      1115
   macro avg       0.43      0.50      0.46      1115
weighted avg       0.75      0.87      0.80      1115



In [18]:
def predict_spam(text):
    cleaned_text = clean_text(text)
    
    seq = tokenizer.texts_to_sequences([cleaned_text])
    pad_seq = pad_sequences(seq, maxlen=max_length, padding='post', truncating='post')
    
    prediction = model.predict(pad_seq, verbose=0)[0][0]
    
    label = "SPAM" if prediction > 0.3 else "HAM (Not Spam)"
    confidence = prediction if prediction > 0.5 else (1 - prediction)
    
    return label, confidence

user_msg = input("Enter the message to analyze: ")
label, conf = predict_spam(user_msg)

print("-" * 30)
print(f"Message Analyzed: '{user_msg}'")
print(f"Prediction: {label}")
print(f"Confidence Score: {conf * 100:.2f}%")
print("-" * 30)

------------------------------
Message Analyzed: 'congratulations! you won iPhone 17 pro max! click the link below to claim it!'
Prediction: SPAM
Confidence Score: 65.00%
------------------------------


In [19]:
def predict_spam(text):
    cleaned_text = clean_text(text)
    
    seq = tokenizer.texts_to_sequences([cleaned_text])
    pad_seq = pad_sequences(seq, maxlen=max_length, padding='post', truncating='post')
    
    prediction = model.predict(pad_seq, verbose=0)[0][0]
    
    label = "SPAM" if prediction > 0.5 else "HAM (Not Spam)"
    confidence = prediction if prediction > 0.5 else (1 - prediction)
    
    return label, confidence

user_msg = input("Enter the message to analyze: ")
label, conf = predict_spam(user_msg)

print("-" * 30)
print(f"Message Analyzed: '{user_msg}'")
print(f"Prediction: {label}")
print(f"Confidence Score: {conf * 100:.2f}%")
print("-" * 30)

------------------------------
Message Analyzed: 'hi! how are you?'
Prediction: HAM (Not Spam)
Confidence Score: 65.00%
------------------------------
